In [1]:
!pip install vllm==0.10.2 datasets pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB ?  0:09:33eta 0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 9.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 26.1 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 55.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 60.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB ?  0:09:41eta 0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.4/819.4 kB 19.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 43.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0

In [2]:
import torch
import torchvision
import torchaudio
import transformers
import datasets
import peft
import vllm

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("vllm:", vllm.__version__)

torch: 2.8.0+cu128
torchvision: 0.23.0+cu128
torchaudio: 2.8.0+cu128
transformers: 4.57.6
datasets: 3.6.0
peft: 0.19.1
vllm: 0.10.2


In [3]:
import json
import re
import pandas as pd
from typing import List, Dict
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

INFO 07-13 14:13:05 [__init__.py:216] Automatically detected platform cuda.


In [4]:
# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/ecommerce-function-calling-datasets-korean", split="train")

In [5]:
# 테스트 비율 설정
test_ratio = 0.2

# 전체 길이와 테스트 데이터 크기 계산
total_len = len(dataset)
test_size = int(total_len * test_ratio)

# 앞에서부터 테스트 데이터, 나머지는 학습 데이터
test_indices = list(range(test_size))
train_indices = list(range(test_size, total_len))

In [6]:
# OpenAI 포맷으로 변환 함수
def format_conversations(sample):
    return {
        "messages": [
            {"role": "system", "content": sample["system_prompt"]},
            *sample["messages"]
        ]
    }

# 분할 및 변환
train_dataset = [format_conversations(dataset[i]) for i in train_indices]
test_dataset = [format_conversations(dataset[i]) for i in test_indices]

# 리스트를 다시 HuggingFace Dataset 객체로 변환
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

# 결과 확인
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")


전체 데이터 분할 결과: Train 311개, Test 77개


In [7]:
def remove_think_blocks(text):
    if text is None:
        return ""

    text = str(text)
    text = text.replace("<think>\n\n</think>\n\n", "")

    return text.strip()

In [8]:
def to_chatml(data):
    """
    data: messages 리스트이거나 {"messages": [...]} 형태의 dict
    반환값: 채팅 템플릿 포맷의 문자열
    """
    # data가 dict이고 'messages' 키가 있으면 messages 리스트를 꺼내고,
    # 아니면 data 자체를 messages 리스트로 간주
    messages = data.get("messages") if isinstance(data, dict) and "messages" in data else data

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    text = remove_think_blocks(text)

    return text

In [9]:
def extract_examples(chatml: str) -> List[Dict[str, str]]:
    """
    채팅 템플릿 문자열에서 각 assistant 응답을 분리하여
    'input'과 'label' 쌍을 생성합니다.
    'input'은 해당 assistant 응답 직전까지의 모든 대화 + '<|im_start|>assistant',
    'label'은 해당 assistant의 응답 내용입니다.
    """
    examples: List[Dict[str, str]] = []
    pattern = re.compile(r'<\|im_start\|>assistant(.*?)(?=<\|im_end\|>)', re.DOTALL)

    for match in pattern.finditer(chatml):
        start_idx = match.start()
        input_text = chatml[:start_idx].strip() + '\n<|im_start|>assistant'
        label_text = match.group(1).strip()

        input_text = remove_think_blocks(input_text)
        label_text = remove_think_blocks(label_text)

        examples.append({
            "input": input_text,
            "label": label_text
        })

    return examples

In [10]:
# Qwen3-4B 로컬 스냅샷 경로에서 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("/workspace/models/Qwen3-4B")

In [11]:
prompt_lst = []
label_lst = []

for item in test_dataset:
    chatml = to_chatml(item)  # 채팅 템플릿 문자열로 변환
    examples = extract_examples(chatml)  # assistant 응답 단위로 분리

    for ex in examples:
        prompt_lst.append(ex['input'])
        label_lst.append(ex['label'])

In [12]:
print(prompt_lst[10])

<|im_start|>system
당신은 상준몰의 AI 상담사입니다. 성심성의껏 상담하십시오.

로그인한 사용자의 현재 ID: U006
오늘 날짜: 2024-02-02

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "add_to_cart", "description": "사용자의 장바구니에 지정된 상품(product_id)과 수량(quantity)을 추가합니다. 동일 상품이 이미 있으면 수량을 증가시키고, 새 항목으로 추가합니다.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string", "description": "장바구니에 상품을 추가할 사용자의 고유 식별자 (예: 'U001')"}, "product_id": {"type": "string", "description": "장바구니에 추가할 상품의 고유 식별자 (예: 'P003')"}, "quantity": {"type": "integer", "description": "추가할 상품 수량 (기본값: 1)", "default": 1, "minimum": 1}}, "required": ["user_id", "product_id"], "additionalProperties": false}}}
{"type": "function", "function": {"name": "view_order_history", "description": "사용자의 전체 주문 내역을 반환합니다. 각 주문에 대해 주문 번호, 주문 일자, 총 결제 금액, 결제 상태, 배송 상태, 택배사, 운송장 번호, 배송 진행 단계, 주문에 포함된 상품명 목록을 제공

In [13]:
print(label_lst[10])

<tool_call>
{"name": "search_product", "arguments": {"keyword": "노트북"}}
</tool_call>


In [14]:
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=2048,
    stop=["<|im_end|>"]
)

In [15]:
llm = LLM(model="qwen-3-4b-function-calling/merged")

INFO 07-13 14:13:08 [utils.py:328] non-default args: {'disable_log_stats': True, 'model': 'qwen-3-4b-function-calling/merged'}
INFO 07-13 14:13:18 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 07-13 14:13:18 [__init__.py:1815] Using max model len 40960
INFO 07-13 14:13:20 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.


The tokenizer you are loading from 'qwen-3-4b-function-calling/merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=2273) INFO 07-13 14:13:21 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=2273) INFO 07-13 14:13:21 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='qwen-3-4b-function-calling/merged', speculative_config=None, tokenizer='qwen-3-4b-function-calling/merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None),

...

(EngineCore_DP0 pid=2273) INFO 07-13 14:13:27 [default_loader.py:268] Loading weights took 1.71 seconds
(EngineCore_DP0 pid=2273) INFO 07-13 14:13:27 [gpu_model_runner.py:2392] Model loading took 7.5552 GiB and 2.025909 seconds
(EngineCore_DP0 pid=2273) INFO 07-13 14:13:33 [backends.py:539] Using cache directory: /root/.cache/vllm/torch_compile_cache/eae1341c1d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2273) INFO 07-13 14:13:33 [backends.py:550] Dynamo bytecode transform time: 5.77 s
(EngineCore_DP0 pid=2273) INFO 07-13 14:13:39 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=2273) INFO 07-13 14:14:03 [backends.py:215] Compiling a graph for dynamic shape takes 29.06 s
(EngineCore_DP0 pid=2273) INFO 07-13 14:14:08 [monitor.py:34] torch.compile takes 34.83 s in total
(EngineCore_DP0 pid=2273) INFO 07-13 14:14:09 [gpu_worker.py:298] Available KV cache memory: 62.32 GiB
(EngineCore_DP0 pid=2273) INFO 07-13 14:14:10 [kv_cache_utils.

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 19.80it/s]


(EngineCore_DP0 pid=2273) INFO 07-13 14:14:14 [gpu_model_runner.py:3118] Graph capturing finished in 4 secs, took 0.54 GiB
(EngineCore_DP0 pid=2273) INFO 07-13 14:14:14 [gpu_worker.py:391] Free memory on device (78.76/79.25 GiB) on startup. Desired GPU memory utilization is (0.9, 71.32 GiB). Actual usage is 7.56 GiB for weight, 1.43 GiB for peak activation, 0.02 GiB for non-torch memory, and 0.54 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=66176537395` to fit into requested memory, or `--kv-cache-memory=74160775168` to fully utilize gpu memory. Current kv cache memory in use is 66918929203 bytes.
(EngineCore_DP0 pid=2273) INFO 07-13 14:14:14 [core.py:218] init engine (profile, create kv cache, warmup model) took 46.65 seconds


(EngineCore_DP0 pid=2273) The tokenizer you are loading from 'qwen-3-4b-function-calling/merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 07-13 14:14:15 [llm.py:295] Supported_tasks: ['generate']
INFO 07-13 14:14:15 [__init__.py:36] No IOProcessor plugins requested by the model


In [16]:
fine_tuned_outputs = llm.generate(prompt_lst, sampling_params)

Adding requests:   0%|          | 0/655 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/655 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [17]:
fine_tuned_text_results = [sample.outputs[0].text.strip() for sample in fine_tuned_outputs]

In [18]:
print(fine_tuned_text_results[10])

<tool_call>
{"name": "search_product", "arguments": {"keyword": "노트북"}}
</tool_call>


In [19]:
df = pd.DataFrame({
    "prompt": prompt_lst,
    "label": label_lst,
    "output": fine_tuned_text_results
})

# 저장
df.to_csv("evaluation_results.csv", index=False, encoding="utf-8-sig")

In [20]:
for label, pred in zip(df['label'].to_list()[:50], df['output'].to_list()[:50]):
    print('레이블 :', label)
    print('--' * 50)
    print('모델의 예측:', pred)
    print('==' * 50)

레이블 : 안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?
----------------------------------------------------------------------------------------------------
모델의 예측: 안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?
레이블 : <tool_call>
{"name": "view_user_profile", "arguments": {"user_id": "U002"}}
</tool_call>
----------------------------------------------------------------------------------------------------
모델의 예측: <tool_call>
{"name": "view_user_profile", "arguments": {"user_id": "U002"}}
</tool_call>
레이블 : 고객님의 주소는 '서울특별시 강남구 테헤란로 123'으로 등록되어 있습니다. 다른 정보가 필요하신가요?
----------------------------------------------------------------------------------------------------
모델의 예측: 고객님의 주소는 '서울특별시 강남구 테헤란로 123'으로 확인됩니다. 다른 정보가 필요하시면 말씀해 주세요.
레이블 : <tool_call>
{"name": "view_order_history", "arguments": {"user_id": "U002"}}
</tool_call>
----------------------------------------------------------------------------------------------------
모델의 예측: <tool_call>
{"name": "view_order_history", "arguments": {"user_id": "U002"}}
</tool_c

In [21]:
def evaluate_function_calls(labels, predictions):
    """
    펑션 콜링 성능을 평가하는 함수
    
    Parameters:
    -----------
    labels : list
        정답 레이블 목록
    predictions : list
        모델이 예측한 결과 목록
        
    Returns:
    --------
    dict
        tool_selection: 함수 이름 일치율
        params_selection: 파라미터 키 일치율 
        params_value_accuracy: 파라미터 값 일치율
        total_samples: 전체 tool_call 샘플 수
    """
    # 결과 저장할 딕셔너리 초기화
    results = {
        'tool_selection': {'correct': 0, 'total': 0},
        'params_selection': {'correct': 0, 'total': 0},
        'params_value_accuracy': {'correct': 0, 'total': 0}
    }
    
    # tool_call 형식만 필터링하기 위한 정규표현식
    tool_call_pattern = re.compile(r'<tool_call>(.*?)</tool_call>', re.DOTALL)
    
    # 전체 샘플 중 tool_call 샘플 수
    tool_call_count = 0
    
    for label, pred in zip(labels, predictions):
        # tool_call 형식인지 확인
        label_match = tool_call_pattern.search(label)
        pred_match = tool_call_pattern.search(pred)
        
        # 레이블이 tool_call이 아니면 건너뛰기
        if not label_match:
            continue
        
        tool_call_count += 1
        
        # 예측이 tool_call 형식이 아니면 모든 지표가 틀린 것으로 처리
        if not pred_match:
            results['tool_selection']['total'] += 1
            results['params_selection']['total'] += 1
            results['params_value_accuracy']['total'] += 1
            continue
        
        # JSON 파싱
        try:
            label_json = json.loads(label_match.group(1))
            pred_json = json.loads(pred_match.group(1))
        except json.JSONDecodeError:
            # JSON 파싱 오류 시 모든 지표가 틀린 것으로 처리
            results['tool_selection']['total'] += 1
            results['params_selection']['total'] += 1
            results['params_value_accuracy']['total'] += 1
            continue
        
        # 1. 함수 이름 일치 여부 (tool_selection)
        results['tool_selection']['total'] += 1
        if label_json.get('name') == pred_json.get('name'):
            results['tool_selection']['correct'] += 1
        
        # 2. 파라미터 키 일치 여부 (params_selection)
        # 개별 파라미터별로 맞고 틀림을 채점
        label_params = set(label_json.get('arguments', {}).keys())
        pred_params = set(pred_json.get('arguments', {}).keys())
        
        # 각 파라미터마다 평가를 위해 모든 파라미터 순회
        for param in label_params:
            results['params_selection']['total'] += 1
            if param in pred_params:
                results['params_selection']['correct'] += 1
        
        # 예측에만 있는 추가 파라미터도 틀린 것으로 평가
        for param in pred_params:
            if param not in label_params:
                results['params_selection']['total'] += 1
                # correct는 증가 안 함 (틀린 것이므로)
        
        # 3. 파라미터 값 일치 여부 (params_value_accuracy)
        # 존재하는 공통 파라미터에 대해서만 값 일치 여부 평가
        label_args = label_json.get('arguments', {})
        pred_args = pred_json.get('arguments', {})
        
        # 공통으로 존재하는 파라미터 키 찾기
        common_params = label_params.intersection(pred_params)
        
        if common_params:  # 공통 파라미터가 있는 경우에만 평가
            results['params_value_accuracy']['total'] += 1
            
            # 공통 파라미터의 값이 모두 일치하는지 확인
            values_match = True
            for key in common_params:
                if label_args.get(key) != pred_args.get(key):
                    values_match = False
                    break
            
            if values_match:
                results['params_value_accuracy']['correct'] += 1
    
    # 최종 결과 계산
    final_results = {}
    for metric, counts in results.items():
        if counts['total'] > 0:
            final_results[metric] = counts['correct'] / counts['total']
        else:
            final_results[metric] = 0.0
    
    final_results['total_samples'] = tool_call_count
    
    return final_results

In [22]:
labels = [
    '안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?',
    '<tool_call>\n{"name": "view_user_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '고객님의 주소는 \'서울특별시 강남구 테헤란로 123\'으로 등록되어 있습니다. 다른 정보가 필요하신가요?',
    '<tool_call>\n{"name": "view_order_history", "arguments": {"user_id": "U002"}}\n</tool_call>'
]

predictions = [
    '안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?',
    '<tool_call>\n{"name": "view_user_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '고객님의 주소는 \'서울특별시 강남구 테헤란로 123\'로 등록되어 있습니다. 다른 정보가 필요하시면 말씀해 주세요.',
    '<tool_call>\n{"name": "view_order_history", "arguments": {"user_id": "U002"}}\n</tool_call>'
]

In [23]:
# 정상 케이스 평가
results = evaluate_function_calls(labels, predictions)
print("정상 케이스 평가 결과:")
for metric, value in results.items():
    if metric != 'total_samples':
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")

정상 케이스 평가 결과:
tool_selection: 100.00%
params_selection: 100.00%
params_value_accuracy: 100.00%
total_samples: 2


In [24]:
# 다른 예시 (에러 케이스)
labels_with_errors = [
    '<tool_call>\n{"name": "view_user_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '<tool_call>\n{"name": "search_product", "arguments": {"keyword": "노트북", "category": "전자기기"}}\n</tool_call>',
    '<tool_call>\n{"name": "check_stock", "arguments": {"product_id": "P001"}}\n</tool_call>'
]

predictions_with_errors = [
    '<tool_call>\n{"name": "view_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '<tool_call>\n{"name": "search_product", "arguments": {"keyword": "노트북"}}\n</tool_call>', 
    '죄송합니다. 재고 확인은 제품 번호가 필요합니다.'
]

In [25]:
print("\n에러 케이스 평가 결과:")
results_with_errors = evaluate_function_calls(labels_with_errors, predictions_with_errors)
for metric, value in results_with_errors.items():
    if metric != 'total_samples':
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")


에러 케이스 평가 결과:
tool_selection: 33.33%
params_selection: 50.00%
params_value_accuracy: 66.67%
total_samples: 3


In [26]:
labels = df['label'].to_list()
preds = df['output'].to_list()

results_with_errors = evaluate_function_calls(labels, preds)
for metric, value in results_with_errors.items():
    if metric != 'total_samples':
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")

tool_selection: 95.41%
params_selection: 92.92%
params_value_accuracy: 94.33%
total_samples: 196
